# Corner held-out far-to-near sweep plots

This notebook only reads the saved training-data sweep CSV and regenerates the figures. It does not import TensorFlow and does not train any models.

Run `ml_32_corner_far_to_near.ipynb` first if `data_amount_sweep_corner_far_to_near_fixed_surrogate.csv` does not exist.

The final section adds a paper-style visualization of training-set growth: each training point is colored by the first training percentage at which it enters the far-to-near sweep, using progressively darker green shades.


In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path("/private/tmp") / "matplotlib-cache"))

import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
import numpy as np
import pandas as pd

# This notebook can be run either from the transmon experiment folder or from the repo root.
HERE = Path.cwd()
EXPERIMENT_RELATIVE = Path("experiments/model_predict_qubit_TransmonCross_Hamiltonian_params")

if (HERE / "metadata" / "qubit-TransmonCross-Hamiltonian_params.json").exists():
    EXPERIMENT_DIR = HERE
elif (HERE / EXPERIMENT_RELATIVE / "metadata" / "qubit-TransmonCross-Hamiltonian_params.json").exists():
    EXPERIMENT_DIR = HERE / EXPERIMENT_RELATIVE
else:
    raise FileNotFoundError(
        "Could not find the transmon-cross metadata file. "
        "Run this notebook from the repo root or from the transmon experiment folder."
    )

METADATA_DIR = EXPERIMENT_DIR / "metadata"
METADATA_PATH = METADATA_DIR / "qubit-TransmonCross-Hamiltonian_params.json"
OUT_PATH = EXPERIMENT_DIR / "data_amount_sweep_corner_far_to_near_fixed_surrogate.csv"
SUMMARY_OUT_PATH = EXPERIMENT_DIR / "data_amount_sweep_corner_far_to_near_fixed_surrogate_summary.csv"
PLOT_DIR = EXPERIMENT_DIR / "plots" / "corner_far_to_near_sweep_fixed_surrogate"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

if not OUT_PATH.exists():
    raise FileNotFoundError(f"Missing {OUT_PATH}. Run ml_32_corner_far_to_near.ipynb first.")

out = pd.read_csv(OUT_PATH)
FRACTIONS = tuple(sorted(float(v) for v in out["fraction"].dropna().unique()))
if not FRACTIONS:
    raise ValueError("No fractions found in sweep CSV")

REQUESTED_TEST_CORNER_UM = np.array([
    float(out["requested_corner_claw_um"].iloc[0]),
    float(out["requested_corner_ground_um"].iloc[0]),
    float(out["requested_corner_cross_um"].iloc[0]),
])
TEST_FRACTION = 0.15
VAL_FRACTION = 0.15
EPS = 1e-12

GREEN = "#3D8B3D"
GREEN_LIGHT = "#E8F5E8"
ORANGE = "#E87A00"
ORANGE_LIGHT = "#FFF4E6"
PURPLE = "#7B68AE"
PURPLE_LIGHT = "#E8E4F0"
BLUE = "#3F6F8B"
GREY = "#B8B8B8"
DARK = "#222222"

paper_style = {
    "text.usetex": False,
    "mathtext.fontset": "cm",
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 9,
    "axes.titlesize": 10.5,
    "axes.titleweight": "normal",
    "axes.labelsize": 9,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.0,
    "axes.labelcolor": DARK,
    "axes.edgecolor": "#888888",
    "xtick.color": DARK,
    "ytick.color": DARK,
    "text.color": DARK,
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.grid": False,
    "grid.color": "#D7D7D7",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}
COLUMN_WIDTH_IN = 3.35

print(f"Loaded {len(out)} sweep rows from {OUT_PATH}")
print("Fractions:", ", ".join(f"{100*f:.0f}%" for f in FRACTIONS))
print("Plot directory:", PLOT_DIR)


In [ ]:
@dataclass
class Scaler:
    min_: np.ndarray
    max_: np.ndarray

    @property
    def range_(self) -> np.ndarray:
        return np.maximum(self.max_ - self.min_, EPS)

    def transform(self, x: np.ndarray) -> np.ndarray:
        return (np.asarray(x, dtype=np.float64) - self.min_) / self.range_


def parse_um(value: object) -> float:
    text = str(value).strip()
    for suffix in ("um", "µm", "μm"):
        if text.endswith(suffix):
            return float(text[: -len(suffix)])
    return float(text)


def load_geometry_um() -> np.ndarray:
    data = json.loads(METADATA_PATH.read_text())
    geometry = []
    for row in data:
        opts = row["design"]["design_options"]
        readout = opts["connection_pads"]["readout"]
        geometry.append([
            parse_um(readout["claw_length"]),
            parse_um(readout["ground_spacing"]),
            parse_um(opts["cross_length"]),
        ])
    return np.asarray(geometry, dtype=np.float64)


def choose_corner_heldout_split(
    geometry_um: np.ndarray,
    requested_corner_um: np.ndarray,
    test_fraction: float = 0.15,
    val_fraction: float = 0.15,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n_rows = len(geometry_um)
    n_test = int(np.ceil(test_fraction * n_rows))
    n_val = int(np.ceil(val_fraction * n_rows))

    geom_min = geometry_um.min(axis=0)
    geom_max = geometry_um.max(axis=0)
    clipped_corner_um = np.clip(requested_corner_um, geom_min, geom_max)

    corner_scaler = Scaler(geom_min, geom_max)
    geometry_scaled = corner_scaler.transform(geometry_um)
    corner_scaled = corner_scaler.transform(clipped_corner_um.reshape(1, -1))[0]

    corner_distance = np.linalg.norm(geometry_scaled - corner_scaled, axis=1)
    near_corner_order = np.argsort(corner_distance)

    corner_idx = near_corner_order[: n_test + n_val]
    rng = np.random.default_rng(42)
    rng.shuffle(corner_idx)

    test_idx = corner_idx[:n_test]
    val_idx = corner_idx[n_test:]
    train_pool_idx = near_corner_order[n_test + n_val :]
    return train_pool_idx, val_idx, test_idx, clipped_corner_um


def rank_training_far_to_near(geometry_scaled: np.ndarray, train_pool_idx: np.ndarray, test_idx: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    train_geom = geometry_scaled[train_pool_idx]
    test_geom = geometry_scaled[test_idx]
    distances = np.sqrt(((train_geom[:, None, :] - test_geom[None, :, :]) ** 2).sum(axis=2))
    train_to_test_distance = distances.min(axis=1)
    far_to_near_order = np.argsort(-train_to_test_distance)
    return far_to_near_order, train_to_test_distance


def subset_for_fraction(fraction: float) -> np.ndarray:
    n_subset = max(1, int(round(fraction * len(train_pool_idx))))
    subset_local = far_to_near_order[:n_subset]
    return train_pool_idx[subset_local]


def training_entry_percentages() -> np.ndarray:
    entry_percent_local = np.full(len(train_pool_idx), np.nan, dtype=float)
    for fraction in FRACTIONS:
        n_subset = max(1, int(round(fraction * len(train_pool_idx))))
        local_idx = far_to_near_order[:n_subset]
        newly_added = np.isnan(entry_percent_local[local_idx])
        entry_percent_local[local_idx[newly_added]] = fraction * 100.0
    entry_percent_local[np.isnan(entry_percent_local)] = max(FRACTIONS) * 100.0

    entry_percent_global = np.full(len(geom_raw_um), np.nan, dtype=float)
    entry_percent_global[train_pool_idx] = entry_percent_local
    return entry_percent_global


def style_3d_axis(ax):
    ax.view_init(elev=22, azim=-55)
    ax.set_xlabel("claw length (µm)")
    ax.set_ylabel("ground spacing (µm)")
    ax.set_zlabel("cross length (µm)")
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_facecolor((1, 1, 1, 0))
        axis.pane.set_edgecolor("#DDDDDD")
        axis._axinfo["grid"]["color"] = (0.84, 0.84, 0.84, 0.55)
        axis._axinfo["grid"]["linewidth"] = 0.45


def progression_colormap():
    levels = np.array([fraction * 100.0 for fraction in FRACTIONS], dtype=float)
    colors = plt.get_cmap("Greens")(np.linspace(0.34, 0.88, len(levels)))
    cmap = ListedColormap(colors)
    if len(levels) == 1:
        bounds = np.array([levels[0] - 0.5, levels[0] + 0.5])
    else:
        mids = (levels[:-1] + levels[1:]) / 2.0
        bounds = np.r_[levels[0] - (mids[0] - levels[0]), mids, levels[-1] + (levels[-1] - mids[-1])]
    return levels, cmap, BoundaryNorm(bounds, cmap.N)


geom_raw_um = load_geometry_um()
train_pool_idx, val_idx, test_idx, clipped_corner_um = choose_corner_heldout_split(
    geom_raw_um,
    REQUESTED_TEST_CORNER_UM,
    TEST_FRACTION,
    VAL_FRACTION,
)
geom_split_scaler = Scaler(
    geom_raw_um[train_pool_idx].min(axis=0),
    geom_raw_um[train_pool_idx].max(axis=0),
)
geom_scaled = geom_split_scaler.transform(geom_raw_um)
far_to_near_order, train_to_test_distance = rank_training_far_to_near(geom_scaled, train_pool_idx, test_idx)
entry_percent_global = training_entry_percentages()

print(f"Total samples: {len(geom_raw_um)}")
print(f"Training pool / validation / test: {len(train_pool_idx)} / {len(val_idx)} / {len(test_idx)}")
print("Requested corner [µm]:", REQUESTED_TEST_CORNER_UM)
print("Used clipped corner [µm]:", clipped_corner_um)


In [ ]:
summary = (
    out.groupby(["training_percent", "n_samples"], as_index=False)
    .agg(
        train_mean=("train_mean_hamiltonian_pct", "mean"),
        train_std=("train_mean_hamiltonian_pct", "std"),
        val_mean=("val_mean_hamiltonian_pct", "mean"),
        val_std=("val_mean_hamiltonian_pct", "std"),
        test_mean=("test_mean_hamiltonian_pct", "mean"),
        test_std=("test_mean_hamiltonian_pct", "std"),
        dist_min=("subset_to_test_nn_distance_min", "mean"),
        dist_median=("subset_to_test_nn_distance_median", "mean"),
        dist_max=("subset_to_test_nn_distance_max", "mean"),
    )
    .sort_values("training_percent")
)
summary.to_csv(SUMMARY_OUT_PATH, index=False)

with plt.rc_context(paper_style):
    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH_IN, 2.45))

    series = [
        ("Training", "train_mean", "train_std", GREEN, GREEN_LIGHT, "o"),
        ("Validation", "val_mean", "val_std", ORANGE, ORANGE_LIGHT, "s"),
        ("Test", "test_mean", "test_std", PURPLE, PURPLE_LIGHT, "^"),
    ]
    x = summary["training_percent"].to_numpy()

    for label, mean_col, std_col, color, fill, marker in series:
        mean = summary[mean_col].to_numpy()
        std = summary[std_col].fillna(0).to_numpy()
        ax.plot(x, mean, marker=marker, markersize=3.7, linewidth=1.45, color=color, label=label, zorder=3)
        ax.fill_between(x, mean - std, mean + std, color=fill, alpha=0.58, linewidth=0, zorder=1)

    ax.set_xticks(x)
    ax.set_xticklabels([f"{v:.0f}" for v in x])
    ax.set_xlabel("Training pool used [%]")
    ax.set_ylabel("Mean Hamiltonian error [%]")
    ax.set_title("Far-to-near data sweep")
    ax.grid(axis="y", linestyle=":", color="#D7D7D7")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_ylim(bottom=0)
    ax.margins(x=0.04)
    ax.legend(loc="upper right", frameon=True, edgecolor="#CCCCCC", facecolor="white", handlelength=1.25, borderpad=0.35, labelspacing=0.35)

    fig.tight_layout(pad=0.35)
    fig.savefig(PLOT_DIR / "corner_far_to_near_learning_curve.pdf", bbox_inches="tight", pad_inches=0.03)
    fig.savefig(PLOT_DIR / "corner_far_to_near_learning_curve.png", dpi=300, bbox_inches="tight", pad_inches=0.03)
    plt.show()

summary


In [ ]:
with plt.rc_context(paper_style):
    fig = plt.figure(figsize=(5.4, 4.3))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(geom_raw_um[train_pool_idx, 0], geom_raw_um[train_pool_idx, 1], geom_raw_um[train_pool_idx, 2], s=10, c=GREY, alpha=0.22, label="training pool", depthshade=False)
    ax.scatter(geom_raw_um[val_idx, 0], geom_raw_um[val_idx, 1], geom_raw_um[val_idx, 2], s=18, c=ORANGE, alpha=0.86, label="validation corner", depthshade=False)
    ax.scatter(geom_raw_um[test_idx, 0], geom_raw_um[test_idx, 1], geom_raw_um[test_idx, 2], s=20, c=PURPLE, alpha=0.90, label="test corner", depthshade=False)
    ax.scatter([clipped_corner_um[0]], [clipped_corner_um[1]], [clipped_corner_um[2]], s=130, marker="*", c=BLUE, edgecolor="white", linewidth=0.8, label="chosen corner", depthshade=False)

    style_3d_axis(ax)
    ax.set_title("Held-out corner split")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "corner_split_3d.pdf", bbox_inches="tight")
    fig.savefig(PLOT_DIR / "corner_split_3d.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
def plot_fraction_3d(fraction: float, save: bool = True) -> None:
    subset_idx = subset_for_fraction(fraction)
    unused_idx = np.setdiff1d(train_pool_idx, subset_idx, assume_unique=False)

    with plt.rc_context(paper_style):
        fig = plt.figure(figsize=(5.6, 4.3))
        ax = fig.add_subplot(111, projection="3d")

        ax.scatter(geom_raw_um[unused_idx, 0], geom_raw_um[unused_idx, 1], geom_raw_um[unused_idx, 2], s=8, c=GREY, alpha=0.13, label="not used yet", depthshade=False)
        ax.scatter(geom_raw_um[subset_idx, 0], geom_raw_um[subset_idx, 1], geom_raw_um[subset_idx, 2], s=13, c=GREEN, alpha=0.76, label=f"training used ({fraction:.0%})", depthshade=False)
        ax.scatter(geom_raw_um[val_idx, 0], geom_raw_um[val_idx, 1], geom_raw_um[val_idx, 2], s=18, c=ORANGE, alpha=0.86, label="validation corner", depthshade=False)
        ax.scatter(geom_raw_um[test_idx, 0], geom_raw_um[test_idx, 1], geom_raw_um[test_idx, 2], s=20, c=PURPLE, alpha=0.90, label="test corner", depthshade=False)
        ax.scatter([clipped_corner_um[0]], [clipped_corner_um[1]], [clipped_corner_um[2]], s=130, marker="*", c=BLUE, edgecolor="white", linewidth=0.8, label="chosen corner", depthshade=False)

        style_3d_axis(ax)
        ax.set_title(f"Geometry coverage at {fraction:.0%} training fraction")
        ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True)
        fig.tight_layout()

        if save:
            stem = f"geometry_3d_fraction_{int(round(fraction * 100)):03d}pct"
            fig.savefig(PLOT_DIR / f"{stem}.pdf", bbox_inches="tight")
            fig.savefig(PLOT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
        plt.show()


def plot_fraction_pairwise(fraction: float, save: bool = True) -> None:
    subset_idx = subset_for_fraction(fraction)
    unused_idx = np.setdiff1d(train_pool_idx, subset_idx, assume_unique=False)
    pairs = [
        (0, 1, "claw length (µm)", "ground spacing (µm)"),
        (0, 2, "claw length (µm)", "cross length (µm)"),
        (1, 2, "ground spacing (µm)", "cross length (µm)"),
    ]

    with plt.rc_context(paper_style):
        fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.65))
        for ax, (i, j, xlabel, ylabel) in zip(axes, pairs):
            ax.scatter(geom_raw_um[unused_idx, i], geom_raw_um[unused_idx, j], s=8, c=GREY, alpha=0.13, label="not used yet")
            ax.scatter(geom_raw_um[subset_idx, i], geom_raw_um[subset_idx, j], s=12, c=GREEN, alpha=0.76, label="training used")
            ax.scatter(geom_raw_um[val_idx, i], geom_raw_um[val_idx, j], s=16, c=ORANGE, alpha=0.86, label="validation corner")
            ax.scatter(geom_raw_um[test_idx, i], geom_raw_um[test_idx, j], s=18, c=PURPLE, alpha=0.90, label="test corner")
            ax.scatter([clipped_corner_um[i]], [clipped_corner_um[j]], s=95, marker="*", c=BLUE, edgecolor="white", linewidth=0.7, label="chosen corner")
            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)
            ax.grid(linestyle=":", color="#D7D7D7", alpha=0.7)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=5, frameon=True, bbox_to_anchor=(0.5, -0.11))
        fig.suptitle(f"Pairwise geometry coverage at {fraction:.0%} training fraction", y=1.03)
        fig.tight_layout()

        if save:
            stem = f"geometry_pairwise_fraction_{int(round(fraction * 100)):03d}pct"
            fig.savefig(PLOT_DIR / f"{stem}.pdf", bbox_inches="tight")
            fig.savefig(PLOT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
        plt.show()


In [ ]:
for fraction in FRACTIONS:
    plot_fraction_3d(fraction)
    plot_fraction_pairwise(fraction)

print(f"Saved fraction-by-fraction geometry plots to {PLOT_DIR}")


In [ ]:
with plt.rc_context(paper_style):
    fig = plt.figure(figsize=(9.0, 5.5))

    for panel_idx, fraction in enumerate(FRACTIONS, start=1):
        subset_idx = subset_for_fraction(fraction)
        ax = fig.add_subplot(2, 4, panel_idx, projection="3d")
        ax.scatter(geom_raw_um[train_pool_idx, 0], geom_raw_um[train_pool_idx, 1], geom_raw_um[train_pool_idx, 2], s=4.5, c=GREY, alpha=0.10, depthshade=False)
        ax.scatter(geom_raw_um[subset_idx, 0], geom_raw_um[subset_idx, 1], geom_raw_um[subset_idx, 2], s=8, c=GREEN, alpha=0.72, depthshade=False)
        ax.scatter(geom_raw_um[test_idx, 0], geom_raw_um[test_idx, 1], geom_raw_um[test_idx, 2], s=9, c=PURPLE, alpha=0.85, depthshade=False)
        ax.set_title(f"{fraction:.0%}")
        ax.set_xlabel("claw")
        ax.set_ylabel("ground")
        ax.set_zlabel("cross")
        ax.view_init(elev=22, azim=-55)
        for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
            axis.pane.set_facecolor((1, 1, 1, 0))
            axis._axinfo["grid"]["color"] = (0.84, 0.84, 0.84, 0.45)
            axis._axinfo["grid"]["linewidth"] = 0.35

    ax_empty = fig.add_subplot(2, 4, 8)
    ax_empty.axis("off")
    ax_empty.scatter([], [], s=20, c=GREEN, label="training used")
    ax_empty.scatter([], [], s=20, c=PURPLE, label="held-out test corner")
    ax_empty.scatter([], [], s=20, c=GREY, alpha=0.5, label="training pool")
    ax_empty.legend(loc="center left", frameon=True)

    fig.suptitle("Far-to-near growth of the training set in geometry space", y=0.98)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(PLOT_DIR / "geometry_3d_all_fractions_panel.pdf", bbox_inches="tight")
    fig.savefig(PLOT_DIR / "geometry_3d_all_fractions_panel.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
def plot_progression_green_shades_3d(save: bool = True) -> None:
    levels, cmap, norm = progression_colormap()
    train_entry = entry_percent_global[train_pool_idx]

    with plt.rc_context(paper_style):
        fig = plt.figure(figsize=(5.2, 4.15))
        ax = fig.add_subplot(111, projection="3d")
        sc = ax.scatter(
            geom_raw_um[train_pool_idx, 0],
            geom_raw_um[train_pool_idx, 1],
            geom_raw_um[train_pool_idx, 2],
            s=13,
            c=train_entry,
            cmap=cmap,
            norm=norm,
            alpha=0.82,
            linewidths=0,
            depthshade=False,
        )
        ax.scatter(geom_raw_um[val_idx, 0], geom_raw_um[val_idx, 1], geom_raw_um[val_idx, 2], s=20, facecolors="none", edgecolors=ORANGE, linewidths=0.85, alpha=0.9, label="validation", depthshade=False)
        ax.scatter(geom_raw_um[test_idx, 0], geom_raw_um[test_idx, 1], geom_raw_um[test_idx, 2], s=22, facecolors="none", edgecolors=PURPLE, linewidths=0.9, alpha=0.95, label="test", depthshade=False)
        ax.scatter([clipped_corner_um[0]], [clipped_corner_um[1]], [clipped_corner_um[2]], s=125, marker="*", c=BLUE, edgecolor="white", linewidth=0.8, label="corner", depthshade=False)

        style_3d_axis(ax)
        ax.set_title("Training-set growth through geometry space")
        ax.legend(loc="upper left", bbox_to_anchor=(1.13, 1.0), frameon=True, borderpad=0.35, labelspacing=0.35)
        cbar = fig.colorbar(sc, ax=ax, fraction=0.035, pad=0.08)
        cbar.set_label("First included at [%]")
        cbar.set_ticks(levels)
        cbar.set_ticklabels([f"{level:.0f}" for level in levels])

        fig.tight_layout()
        if save:
            fig.savefig(PLOT_DIR / "geometry_3d_progression_green_shades.pdf", bbox_inches="tight")
            fig.savefig(PLOT_DIR / "geometry_3d_progression_green_shades.png", dpi=300, bbox_inches="tight")
        plt.show()


def plot_progression_green_shades_pairwise(save: bool = True) -> None:
    levels, cmap, norm = progression_colormap()
    train_entry = entry_percent_global[train_pool_idx]
    pairs = [
        (0, 1, "claw length (µm)", "ground spacing (µm)"),
        (0, 2, "claw length (µm)", "cross length (µm)"),
        (1, 2, "ground spacing (µm)", "cross length (µm)"),
    ]

    with plt.rc_context(paper_style):
        fig, axes = plt.subplots(1, 3, figsize=(7.9, 2.75))
        sc = None
        for ax, (i, j, xlabel, ylabel) in zip(axes, pairs):
            sc = ax.scatter(geom_raw_um[train_pool_idx, i], geom_raw_um[train_pool_idx, j], s=12, c=train_entry, cmap=cmap, norm=norm, alpha=0.82, linewidths=0)
            ax.scatter(geom_raw_um[val_idx, i], geom_raw_um[val_idx, j], s=16, facecolors="none", edgecolors=ORANGE, linewidths=0.8, alpha=0.9)
            ax.scatter(geom_raw_um[test_idx, i], geom_raw_um[test_idx, j], s=18, facecolors="none", edgecolors=PURPLE, linewidths=0.85, alpha=0.95)
            ax.scatter([clipped_corner_um[i]], [clipped_corner_um[j]], s=90, marker="*", c=BLUE, edgecolor="white", linewidth=0.7)
            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)
            ax.grid(linestyle=":", color="#D7D7D7", alpha=0.65)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        fig.suptitle("Far-to-near training-set growth", y=0.96)
        fig.subplots_adjust(left=0.075, right=0.855, bottom=0.23, top=0.80, wspace=0.55)
        cax = fig.add_axes([0.89, 0.24, 0.018, 0.52])
        cbar = fig.colorbar(sc, cax=cax)
        cbar.set_label("First included at [%]", labelpad=8)
        cbar.set_ticks(levels)
        cbar.set_ticklabels([f"{level:.0f}" for level in levels])

        if save:
            fig.savefig(PLOT_DIR / "geometry_pairwise_progression_green_shades.pdf", bbox_inches="tight")
            fig.savefig(PLOT_DIR / "geometry_pairwise_progression_green_shades.png", dpi=300, bbox_inches="tight")
        plt.show()


plot_progression_green_shades_3d()
plot_progression_green_shades_pairwise()
